<a href="https://colab.research.google.com/github/Abdallah387/medical_diagnosis_ai/blob/main/medical_diagnosis_ai_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries
!pip install -U transformers datasets torch scikit-learn pandas==2.2.2 accelerate sentencepiece

# Import libraries
import pandas as pd
import torch
from datasets import Dataset
from transformers import BertTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import json

# Load dataset
data = pd.read_csv("/content/medical_dataset_8000_diseases_augmented.csv")

# Map condition to numbers
unique_conditions = data['condition'].unique()
label_map = {condition: i for i, condition in enumerate(unique_conditions)}
data['labels'] = data['condition'].map(label_map)

num_labels = len(unique_conditions)

# Save labels.json
with open("labels.json", "w") as f:
    json.dump(label_map, f)

# Convert to HuggingFace dataset
dataset = Dataset.from_pandas(data)

# BioBERT model
model_name = "monologg/biobert_v1.1_pubmed"
tokenizer = BertTokenizer.from_pretrained(model_name)

# Tokenization
def tokenize(example):
    tokens = tokenizer(
        example["symptoms"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    tokens["labels"] = example["labels"]
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# Train-test split
train_test = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test["train"]
test_dataset = train_test["test"]

# Load BioBERT with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train
trainer.train()

# Save model, tokenizer, and labels
model.save_pretrained("medical_model")
tokenizer.save_pretrained("medical_model")

# Save labels.json inside folder
with open("medical_model/labels.json", "w") as f:
    json.dump(label_map, f)

# Zip the model folder
!zip -r medical_model.zip medical_model